# Theorem 3: The Late-Game Timeout

> **Based on NBA play-by-play data from 2019–2024, calling a timeout when
> trailing by 1–3 (or tied) with 20–50 seconds remaining does not
> consistently improve win rate — results are mixed across time buckets.**

This notebook reproduces the data collection and visualisation for Theorem 3.
Run each cell in order. Data is saved to `data/processed/theorem3_sweep.csv`
so that plots can be regenerated without re-running the full pipeline.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────────────
import sys
import pathlib

# Make sure the project root is on the path so src.* imports work.
ROOT = pathlib.Path().resolve().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
matplotlib.use("Agg")  # headless; swap to "inline" in a live Jupyter kernel
import matplotlib.pyplot as plt
import numpy as np

PROCESSED_DIR = ROOT / "data" / "processed"
IMAGES_DIR    = ROOT / "docs" / "assets" / "images"

print("Project root  :", ROOT)
print("Processed dir :", PROCESSED_DIR)

## Step 1 – Collect data

Run `collect()` to scan the historical play-by-play log for close-game
possessions (home team trailing by 1–3 or tied, 20–50 s remaining)
and save win rates for the **timeout** and **play-on** strategies
to `theorem3_sweep.csv`.

> *Skip this cell if `theorem3_sweep.csv` already exists in `data/processed/`.*

In [ ]:
from src.theorems.theorem3 import collect

csv_path = collect(out_dir=PROCESSED_DIR, processed_dir=PROCESSED_DIR)
print("Saved:", csv_path)

## Step 2 – Load sweep data

`load_sweep()` reads the CSV and returns a list of per-second-bucket
dictionaries with `ev_timeout`, `ev_play_on`, and `ev_gain` fields.

In [ ]:
from src.theorems.theorem3 import load_sweep

sweep = load_sweep(PROCESSED_DIR)

# Preview all rows
print(f"{'seconds':>10} {'ev_timeout':>12} {'ev_play_on':>12} {'ev_gain':>10} {'optimal':>12}")
print("-" * 62)
for row in sweep:
    print(
        f"{row['seconds_remaining']:>10} "
        f"{row['ev_timeout']:>12.4f} "
        f"{row['ev_play_on']:>12.4f} "
        f"{row['ev_gain']:>10.4f} "
        f"{str(row['timeout_is_optimal']):>12}"
    )

## Step 3 – Reproduce the plot

`plot()` generates the two-panel figure:

* **Top panel** – absolute historical win percentage for Timeout vs. Play On.
* **Bottom panel** – win-percentage gain from calling a timeout (green = timeout
  is better, red = playing on is better). A dotted vertical line marks the
  crossover point if one exists.

In [ ]:
from src.theorems.theorem3 import plot

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

fig_path = plot(processed_dir=PROCESSED_DIR, images_dir=IMAGES_DIR)
print("Saved figure:", fig_path)

## Step 4 – Display the plot inline

In [ ]:
from src.theorems.theorem3 import load_sweep, FONT_FAMILY

sweep      = load_sweep(PROCESSED_DIR)
seconds    = [r["seconds_remaining"] for r in sweep]
ev_timeout = [r["ev_timeout"]         for r in sweep]
ev_play_on = [r["ev_play_on"]         for r in sweep]
ev_gain    = [r["ev_gain"]            for r in sweep]

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top panel
ax1 = axes[0]
ax1.plot(seconds, ev_timeout, color="#E63946", linewidth=2.2, label="Timeout")
ax1.plot(seconds, ev_play_on, color="#457B9D", linewidth=2.2, label="Play On")
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax1.set_ylabel("Historical Win Percentage")
ax1.set_title(
    "Theorem 3: The Late-Game Timeout\nHistorical Win Percentage: Timeout vs. Play On",
    fontweight="bold",
)
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

# Bottom panel
ax2 = axes[1]
gain_arr = np.array(ev_gain)
colors   = np.where(gain_arr >= 0, "#2DC653", "#E63946")
ax2.bar(seconds, gain_arr, color=colors, width=1.6, alpha=0.85)
ax2.axhline(0, color="black", linewidth=1.0)
ax2.set_xlabel("Seconds Remaining")
ax2.set_ylabel("Historical Win % Gain from Timeout (pp)")
ax2.set_title("Win % Gain: Timeout − Play On  (green = timeout is better)")
ax2.grid(True, alpha=0.3, axis="y")

# Mark crossover if it exists
crossover_seconds = None
for i in range(1, len(ev_gain)):
    if ev_gain[i - 1] < 0 and ev_gain[i] >= 0:
        crossover_seconds = seconds[i]
        break
    if ev_gain[i - 1] >= 0 and ev_gain[i] < 0:
        crossover_seconds = seconds[i - 1]
        break

if crossover_seconds is not None:
    ax2.axvline(
        crossover_seconds, color="black", linewidth=1.4, linestyle=":", alpha=0.7,
        label=f"Crossover ≈ {crossover_seconds}s",
    )
    ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## Step 5 – Generate the Markdown documentation

In [ ]:
from src.theorems.theorem3 import generate_doc

DOCS_DIR = ROOT / "docs"
doc_path = generate_doc(processed_dir=PROCESSED_DIR, docs_dir=DOCS_DIR)
print("Saved doc:", doc_path)